# Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import re

# Create Cleaned Data Folder

In [2]:
os.makedirs("data/cleaned", exist_ok=True)

# Load All CSV Files

In [3]:
customers = pd.read_csv("data/raw/customers.csv")
products = pd.read_csv("data/raw/products.csv")
orders = pd.read_csv("data/raw/orders.csv")
order_items = pd.read_csv("data/raw/order_items.csv")

# Check Dataset Information

In [4]:
print("Customers")
print(customers.info())

print("\nProducts")
print(products.info())

print("\nOrders")
print(orders.info())

print("\nOrder Items")
print(order_items.info())

Customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        500 non-null    int64 
 1   customer_name      500 non-null    object
 2   email              500 non-null    object
 3   registration_date  500 non-null    object
 4   customer_type      500 non-null    object
dtypes: int64(1), object(4)
memory usage: 19.7+ KB
None

Products
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    object 
 2   category      500 non-null    object 
 3   subcategory   500 non-null    object 
 4   cost_price    500 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 19.7+ KB
None

Orders
<class 'pa

# PART 1: Clean Customers

In [7]:
# Check Duplicate Records
print("Duplicate Customers :", customers.duplicated().sum())

Duplicate Customers : 0


In [8]:
# Remove Duplicate Records
customers = customers.drop_duplicates()

In [9]:
# Check Missing Values
print(customers.isnull().sum())

customer_id          0
customer_name        0
email                0
registration_date    0
customer_type        0
dtype: int64


In [10]:
# Validate Email Addresses
def valid_email(email):

    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    if pd.isna(email):
        return False

    return re.match(pattern, str(email)) is not None

In [11]:
# Find Invalid Emails
invalid_email_rows = customers[
    ~customers["email"].apply(valid_email)
]

print(invalid_email_rows)

     customer_id     customer_name         email registration_date  \
47            48       Angel Hicks     xyz@yahoo        2024-08-29   
49            50  Jennifer Roberts     xyz@yahoo        2026-07-03   
50            51  Bethany Browning  abcgmail.com        2024-02-23   
87            88    Richard Martin  abcgmail.com        2025-12-03   
97            98  Veronica Delgado     xyz@yahoo        2026-06-14   
163          164         Ryan Ford     xyz@yahoo        2023-07-13   
325          326    Jeremy Hancock     xyz@yahoo        2023-09-20   
342          343         Chad Cook  abcgmail.com        2024-06-10   
430          431        Terri Bond  abcgmail.com        2025-07-30   
491          492      David Greene  abcgmail.com        2023-07-15   

    customer_type  
47        PREMIUM  
49            VIP  
50        REGULAR  
87        PREMIUM  
97        PREMIUM  
163           VIP  
325       REGULAR  
342       PREMIUM  
430           VIP  
491       PREMIUM  


In [12]:
# Replace Invalid Emails
for index in invalid_email_rows.index:

    customer_name = customers.loc[index, "customer_name"]

    username = customer_name.lower().replace(" ", ".")

    customers.loc[index, "email"] = username + "@gmail.com"

In [13]:
# Verify Again
customers[
    ~customers["email"].apply(valid_email)
]

,customer_id,customer_name,email,registration_date,customer_type


# PART 2: Clean Products

In [14]:
# Check Duplicate Products
print(products.duplicated().sum())

0


In [15]:
# Remove Duplicates
products = products.drop_duplicates()

In [16]:
# Remove Extra Spaces
products["product_name"] = products["product_name"].str.strip()

In [17]:
# Convert to Proper Case
products["product_name"] = products["product_name"].str.title()

In [18]:
# Check Result
products.head()

,product_id,product_name,category,subcategory,cost_price
0,1,Sofa,Home,Sofa,2727.77
1,2,Monitor,Electronics,Monitor,3061.23
2,3,Table,Home,Table,3690.30
3,4,Monitor,Electronics,Monitor,257.30
4,5,Sofa,Home,Sofa,552.80


# PART 3: Clean Orders

In [19]:
# Check Missing Values
orders.isnull().sum()

order_id        0
customer_id    50
order_date      0
status          0
region_code     0
dtype: int64

In [20]:
# Remove Duplicate Orders
orders = orders.drop_duplicates()

In [21]:
# Fill Missing Customer IDs
orders["customer_id"] = orders["customer_id"].fillna(
    orders["customer_id"].mode()[0]
)

In [22]:
# Convert Customer ID to Integer
orders["customer_id"] = orders["customer_id"].astype(int)

In [23]:
# Convert Order Date
def convert_date(date):

    try:
        return pd.to_datetime(date)
    except:
        try:
            return pd.to_datetime(date, format="%d-%m-%Y")
        except:
            return pd.NaT

In [24]:
orders["order_date"] = orders["order_date"].apply(convert_date)

In [26]:
# Check Invalid Dates
orders["order_date"].isnull().sum()

np.int64(0)

In [27]:
orders = orders.dropna(subset=["order_date"])

# PART 4: Clean Order Items

In [28]:
# Remove Duplicates
order_items = order_items.drop_duplicates()

In [29]:
# Find Negative Quantity
order_items[
    order_items["quantity"] < 0
]

,item_id,order_id,product_id,quantity,unit_price,discount_percent
46,47,629,108,-4,4530.63,38
52,53,520,185,-4,10226.30,2
63,64,168,357,-4,3330.04,41
84,85,542,89,-3,3118.96,20
192,193,969,462,-3,4021.20,35
...,...,...,...,...,...,...
2841,2842,193,249,-4,13662.98,45
2849,2850,524,324,-4,6719.35,30
2860,2861,855,423,-1,3350.98,11
2881,2882,51,91,-3,8748.98,7


In [30]:
# Convert Negative Quantity to Positive
order_items["quantity"] = order_items["quantity"].abs()

In [31]:
order_items[
    order_items["quantity"] < 0
]

,item_id,order_id,product_id,quantity,unit_price,discount_percent


# PART 5: Referential Integrity

In [32]:
# Find Invalid Order IDs
invalid_orders = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

invalid_orders

,item_id,order_id,product_id,quantity,unit_price,discount_percent
946,947,2001,448,1,4802.65,0
1388,1389,2002,18,2,395.08,31


In [33]:
# Remove Invalid Rows
order_items = order_items[
    order_items["order_id"].isin(orders["order_id"])
]

In [34]:
order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

,item_id,order_id,product_id,quantity,unit_price,discount_percent


# PART 6: Final Dataset Information

In [36]:
print(customers.shape)

print(products.shape)

print(orders.shape)

print(order_items.shape)

(500, 5)
(500, 5)
(1000, 5)
(2998, 6)


# PART 7: Save Cleaned CSV Files

In [37]:
customers.to_csv(
    "data/cleaned/customers_clean.csv",
    index=False
)

products.to_csv(
    "data/cleaned/products_clean.csv",
    index=False
)

orders.to_csv(
    "data/cleaned/orders_clean.csv",
    index=False
)

order_items.to_csv(
    "data/cleaned/order_items_clean.csv",
    index=False
)

# PART 8: Verify Files

In [38]:
os.listdir("data/cleaned")

['customers_clean.csv',
 'orders_clean.csv',
 'order_items_clean.csv',
 'products_clean.csv']

# PART 9: Final Check

In [39]:
print(customers.head())

print(products.head())

print(orders.head())

print(order_items.head())

   customer_id     customer_name                        email  \
0            1       Tammy Bryan  rodgersjonathan@example.com   
1            2      David Daniel      christine14@example.net   
2            3  Christopher Webb          ydalton@example.net   
3            4      Cody Jimenez     allenjeffrey@example.net   
4            5      Rodney Moore       dawnwillis@example.net   

  registration_date customer_type  
0        2024-10-05       REGULAR  
1        2024-08-06       REGULAR  
2        2025-04-10       PREMIUM  
3        2024-03-16           VIP  
4        2024-02-04       PREMIUM  
   product_id product_name     category subcategory  cost_price
0           1         Sofa         Home        Sofa     2727.77
1           2      Monitor  Electronics     Monitor     3061.23
2           3        Table         Home       Table     3690.30
3           4      Monitor  Electronics     Monitor      257.30
4           5         Sofa         Home        Sofa      552.80
   order_